# Week 2: Advanced Pandas Data Wrangling
## Notebook 08: Apache Arrow Zero-Copy IPC & Polars Lazy Interoperability

### 1. Conceptual Intuition & Theoretical Background
Traditional Pandas stores strings as Python object pointers (`np.object_`), which creates extreme memory bloat: each string requires a full 28-50 byte Python string object header plus an 8-byte pointer.

**Apache Arrow** introduces a standardized, language-agnostic **columnar in-memory format**:
- Continuous byte buffer for string data.
- Offset array specifying where each string starts and ends.
- Validity bitmap tracking null values without storing sentinel objects.

$$\text{SIMD Vectorization Speedup} = \frac{\text{Bit Width of Vector Register (e.g. 512)}}{\text{Dtype Width (e.g. 32)}} = 16\times$$

---

### 2. Scratch Python Implementation: Simulating Arrow Columnar Storage

In [ ]:
import numpy as np
import pandas as pd
import sys

# 1. Compare Object Memory vs Flat Arrow-style Buffer
sample_strings = ["user_" + str(i % 100) for i in range(100000)]

# Standard Pandas Object Series
s_obj = pd.Series(sample_strings, dtype="object")
mem_obj = s_obj.memory_usage(deep=True)

# Simulated Arrow Columnar Buffer (Offsets + Packed ASCII Bytes)
distinct_bytes = bytearray()
offsets = [0]
for s in sample_strings:
    b = s.encode('ascii')
    distinct_bytes.extend(b)
    offsets.append(len(distinct_bytes))

offsets_arr = np.array(offsets, dtype=np.int32)
bytes_arr = np.frombuffer(distinct_bytes, dtype=np.uint8)

mem_arrow_sim = offsets_arr.nbytes + bytes_arr.nbytes

print(f"Pandas Object Series Memory:      {mem_obj / (1024 * 1024):.2f} MB")
print(f"Arrow-Style Columnar Buffer Memory: {mem_arrow_sim / (1024 * 1024):.2f} MB")
print(f"Memory Reduction:                {(1 - mem_arrow_sim / mem_obj) * 100:.1f}%")

# 2. Simulated Polars Lazy Query Optimization (Predicate Pushdown)
class SimulatedLazyFrame:
    def __init__(self, data):
        self.data = data
        self.operations = []
        
    def filter(self, condition_func, col_name, predicate_desc):
        # Register filter in computation graph
        self.operations.append(('filter', condition_func, col_name, predicate_desc))
        return self
        
    def select(self, columns):
        self.operations.append(('select', columns))
        return self
        
    def collect(self):
        print("\n--- Query Optimizer Plan ---")
        # Optimization: Pushdown filters before projecting columns
        filters = [op for op in self.operations if op[0] == 'filter']
        selects = [op for op in self.operations if op[0] == 'select']
        
        df = self.data.copy()
        for f in filters:
            print(f"Pushing down predicate: {f[3]} on column '{f[2]}'")
            df = df[f[1](df[f[2]])]
            
        if selects:
            cols = selects[-1][1]
            print(f"Projecting required columns: {cols}")
            df = df[cols]
            
        return df

raw_df = pd.DataFrame({
    'id': np.arange(1000),
    'category': np.random.choice(['A', 'B', 'C'], 1000),
    'revenue': np.random.uniform(10.0, 500.0, 1000)
})

lazy = SimulatedLazyFrame(raw_df)
result = lazy.select(['id', 'revenue']).filter(lambda x: x > 250.0, 'revenue', 'revenue > 250').collect()
print(f"Optimized Query Result Rows: {len(result)}")

### 3. Step-by-Step Code Walkthrough
1. **Arrow In-Memory Structure**: We pack all strings into a single contiguous byte array and maintain 32-bit integer offsets, eliminating per-string Python object overhead.
2. **Predicate Pushdown**: The query optimizer analyzes the execution plan before execution and evaluates filter operations before expensive column transformations.

---

### 4. Industry Applications & Trade-offs
- **High-Throughput Analytics:** Polars and Pandas with PyArrow backend allow handling datasets larger than RAM with multi-threaded streaming queries.
- **Trade-off:** PyArrow strings are immutable; modifying individual characters requires rebuilding the offset array.

---

### 5. Top Interview Questions & Answers
**Q1: What does "Zero-Copy Data Sharing" mean between Pandas, Arrow, and PyTorch?**
> **Answer:** Zero-copy means multiple frameworks access the same physical memory addresses via standardized buffer pointers without copying data across memory regions, achieving microsecond-level transfer times.

**Q2: What is the difference between Eager execution and Lazy evaluation in data wrangling?**
> **Answer:** Eager execution (standard Pandas) evaluates each step immediately. Lazy evaluation (Polars LazyFrame, Spark) constructs a Directed Acyclic Graph (DAG) and optimizes the query (filter pushdown, projection pruning) prior to execution.